## Generate CSV File for satellite orbits

In [ ]:
import pylupnt as pnt
import os
import numpy as np
from src.setup_ionoenv import setup_lcrns_sats

In [ ]:
n_orbit = 6
dt = 1.0
sat_id = 0

t_tai, rv_mci, rv_ecef, N_sc = setup_lcrns_sats(
    n_orbit, dt, savefig=False, overwrite=False
)
rv_mci = rv_mci[sat_id]

tspan = t_tai - t_tai[0]

print("Converting to ECEF...")
rv_ecef = rv_ecef[sat_id]
print("Converting to GCRF...")
rv_gcrf = pnt.convert_frame(t_tai, rv_mci, pnt.MOON_CI, pnt.GCRF)
print("Converting to PA...")
rv_pa = pnt.convert_frame(t_tai, rv_mci, pnt.MOON_CI, pnt.MOON_PA)

## Save to H5Easy

In [ ]:
basepath = pnt.get_output_dir()
orbdir = os.path.join(basepath, "iono_delay", "orbits", "h5")
if not os.path.exists(orbdir):
    os.makedirs(orbdir)

# save to H5
orbfile_h5 = os.path.join(
    orbdir, f"lcrns_sat{sat_id:01d}_norbit{n_orbit}_dt{dt:.0f}.h5"
)

import h5py

with h5py.File(orbfile_h5, "w") as f:
    f.create_dataset("/t_tai", data=t_tai)
    f.create_dataset("/tspan", data=tspan)
    f.create_dataset("/posvel_rx_mci", data=rv_mci)
    f.create_dataset("/posvel_rx_ecef", data=rv_ecef)
    f.create_dataset("/posvel_rx_gcrf", data=rv_gcrf)
    f.create_dataset("/posvel_rx_pa", data=rv_pa)
print(f"Saved orbit data to {orbfile_h5}")

In [ ]:
# Load and verify
with h5py.File(orbfile_h5, "r") as f:
    t_tai_loaded = f["/t_tai"][:]
    rv_mci_loaded = f["/posvel_rx_mci"][:]
    rv_ecef_loaded = f["/posvel_rx_ecef"][:]
    rv_gcrf_loaded = f["/posvel_rx_gcrf"][:]
    rv_pa_loaded = f["/posvel_rx_pa"][:]

# plot to verify
from plotly import graph_objects as go

fig = go.Figure()
pnt.plot.plot_orbits(fig, rv_mci_loaded, color="red")  # [N, t, 3]
# pnt.plot.plot_orbits(fig, rv_pa_loaded, color="blue")
pnt.plot.plot_body(
    fig,
    pnt.MOON,
    size_factor=2,
    alpha=0.5,
)
pnt.plot.set_view(fig, -80, 20, 2.5)
fig.update_layout(showlegend=True, width=400, height=400)
fig.show()